In [1]:
import numpy as np
import open3d as o3d
import laspy  as lp
import matplotlib.pyplot as plt

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
data_path = r'../Data/bunny.ply'
pcd = o3d.io.read_point_cloud(data_path)
pcd_center = pcd.get_center()
pcd.translate(-pcd_center)
o3d.visualization.draw_geometries([pcd])

# Outlier Filter

In [3]:
nn = 16
std_multiplier = 10
filtered_pcd = pcd.remove_statistical_outlier(nn , std_multiplier)
outliers = pcd.select_by_index(filtered_pcd[1] , invert = True)
outliers.paint_uniform_color([1,0,0])
filtered_pcd = filtered_pcd[0]
print(filtered_pcd)

PointCloud with 35947 points.


# Voxel Sampling (We've done it before)

In [4]:
voxel_size = 0.001
pcd_downsampled = filtered_pcd.voxel_down_sample(voxel_size)
print(pcd_downsampled)
o3d.visualization.draw_geometries([pcd_downsampled])

PointCloud with 34583 points.


# Estimating normals


In [5]:
nn_distance = np.mean(pcd_downsampled.compute_nearest_neighbor_distance())
radius_normal = nn_distance * 5

pcd_downsampled.estimate_normals(
    search_param=o3d.geometry.KDTreeSearchParamHybrid(
        radius=radius_normal, max_nn=16
    ), 
    fast_normal_computation=True
)

pcd_downsampled.paint_uniform_color([0.6, 0.6, 0.6])
o3d.visualization.draw_geometries([pcd_downsampled, outliers])

[Open3D WARNING] The number of points is 0 when creating axis-aligned bounding box.


# Ransac

In [6]:
distance_threshold = 0.01   # max point-to-plane distance to count as inlier
ransac_n = 3                # min points to fit a plane (3 = minimal)
num_iterations = 1000       # how many random trials

plane_model, inliers = pcd_downsampled.segment_plane(
    distance_threshold=distance_threshold,
    ransac_n=ransac_n,
    num_iterations=num_iterations
)

[a, b, c, d] = plane_model
print(f"Plane equation: {a:.4f}x + {b:.4f}y + {c:.4f}z + {d:.4f} = 0")

# Separate plane from the rest
inlier_cloud = pcd_downsampled.select_by_index(inliers)
outlier_cloud = pcd_downsampled.select_by_index(inliers, invert=True)

inlier_cloud.paint_uniform_color([1.0, 0.0, 0.0])   # plane → red
outlier_cloud.paint_uniform_color([0.6, 0.6, 0.6])  # rest  → gray

print(f"Plane inliers : {len(inliers)}")
print(f"Remaining pts : {len(pcd_downsampled.points) - len(inliers)}")

o3d.visualization.draw_geometries([inlier_cloud, outlier_cloud])

Plane equation: 0.2773x + 0.4458y + 0.8511z + -0.0165 = 0
Plane inliers : 9455
Remaining pts : 25128


# Multi order ransac

In [7]:
# ── Multi-Order RANSAC Plane Segmentation ─────────────────────────────────────
distance_threshold = 0.01
ransac_n = 3
num_iterations = 1000
max_planes = 5          # maximum number of planes to extract
min_inliers = 100       # stop if a found plane has fewer points than this

remaining = pcd_downsampled  # start with the full downsampled cloud
planes = []
plane_clouds = []

# Generate distinct colors for each plane
colors = [
    [1.0, 0.0, 0.0],   # red
    [0.0, 1.0, 0.0],   # green
    [0.0, 0.0, 1.0],   # blue
    [1.0, 0.5, 0.0],   # orange
    [0.5, 0.0, 1.0],   # purple
]

for i in range(max_planes):
    if len(remaining.points) < ransac_n:
        print(f"Too few points left ({len(remaining.points)}), stopping.")
        break

    plane_model, inliers = remaining.segment_plane(
        distance_threshold=distance_threshold,
        ransac_n=ransac_n,
        num_iterations=num_iterations
    )

    if len(inliers) < min_inliers:
        print(f"Plane {i+1}: only {len(inliers)} inliers (< {min_inliers}), stopping.")
        break

    [a, b, c, d] = plane_model
    print(f"Plane {i+1}: {a:.4f}x + {b:.4f}y + {c:.4f}z + {d:.4f} = 0  |  inliers: {len(inliers)}")

    # Extract the plane and color it
    plane_cloud = remaining.select_by_index(inliers)
    plane_cloud.paint_uniform_color(colors[i % len(colors)])
    plane_clouds.append(plane_cloud)
    planes.append(plane_model)

    # Remove inliers — the remainder feeds the next iteration
    remaining = remaining.select_by_index(inliers, invert=True)

# Whatever is left after all planes are stripped
remaining.paint_uniform_color([0.6, 0.6, 0.6])

print(f"\nExtracted {len(planes)} planes, {len(remaining.points)} points remaining.")
o3d.visualization.draw_geometries(plane_clouds + [remaining])

Plane 1: 0.1827x + -0.1047y + 0.9776z + 0.0238 = 0  |  inliers: 9439
Plane 2: 0.0317x + 0.1778y + 0.9836z + -0.0288 = 0  |  inliers: 8629
Plane 3: 0.0740x + 0.1780y + 0.9812z + -0.0098 = 0  |  inliers: 5549
Plane 4: 0.0111x + 0.2259y + 0.9741z + 0.0148 = 0  |  inliers: 4213
Plane 5: 0.5171x + 0.3961y + 0.7587z + 0.0351 = 0  |  inliers: 2981

Extracted 5 planes, 3772 points remaining.


# 3d Eculidean Refinement - DBSCAN


In [8]:
# ── DBSCAN Clustering on remaining non-plane points ───────────────────────────
labels = np.array(remaining.cluster_dbscan(eps=0.05, min_points=5))

max_labels = labels.max()                          # fix 1: was np.max(remaining)
print(f"Number of clusters: {max_labels + 1}")     # (noise = -1 is excluded)

colors = plt.get_cmap('tab20')(labels / (max_labels if max_labels > 0 else 1))
colors[labels < 0] = 0                             # paint noise black
remaining.colors = o3d.utility.Vector3dVector(colors[:, :3])

o3d.visualization.draw_geometries(plane_clouds + [remaining])

Number of clusters: 1
